# P04 道具の設計 ― モデルは正しい道具を選べるか

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

6 サーバ・17 個の道具を見せて、40 問それぞれで「最初に呼ぶ道具」を 1 回だけ選ばせ、正解と照合します。説明をあいまいにする・名前を番号にする・分野の道具だけ見せる、で何が変わるかを比べます。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. 問いを見る
40 問のうち 38 問は「最初に呼ぶべき道具」が決まっていて、2 問は道具が要らない問いです。どれも架空の会社の問いです。

In [ ]:
from collections import Counter
from tool_variants import load_questions, domain
qs = load_questions()
print("分野ごとの問いの数:", dict(Counter(domain(q) if q["first_tool"] else "道具なし" for q in qs)))
for q in qs[:2] + qs[12:13] + qs[-1:]:
    print(f"{q['id']} {q['question']}  → 最初の道具: {q['first_tool'] or '道具なし'}")

## 2. 道具の欄の大きさを見る
道具の欄 (tools) は毎回の依頼に入るので、そのまま入力の量になります。4 通りの字数と、トークンの目安を比べます (本物の数は応答の usage で見ます)。

In [ ]:
from run_tool_design import amain as tool_design
with safe():
    r = await tool_design(["--describe"])

## 3. 2 問だけ選ばせる
1 問につき依頼を 1 回送り、応答の最初の tool_use だけを読みます。道具は実行しません。

In [ ]:
with safe():
    r = await tool_design(["--only", "TS-01,TS-39", "--show"])

## 4. 40 問を元の説明で
「最初の道具」が正解と同じ数・道具を使わなかった数・どの分野をどこと取り違えたか、を出します。

In [ ]:
with safe():
    r = await tool_design(["--all"])

## 5. 4 通りを比べる
あいまいな説明 (vague)・番号の名前 (names)・分野の道具だけ (subset) を、元の説明 (good) と並べます。mock の数字は性能の証拠になりません。

In [ ]:
with safe():
    r = await tool_design(["--all", "--variant", "all"])

## 6. 同じ問いを 3 通りの見せ方で
在庫・申請の状況・契約の一覧の問いを、元の説明・あいまいな説明・番号の名前で 1 問ずつ見ます。

In [ ]:
with safe():
    r = await tool_design(["--only", "TS-13,TS-20,TS-27", "--variant", "good,vague,names", "--show"])

## 7. 講師の記録を読む
本物の Claude で 4 通りを回した結果は `out/summary_compare.json` に残します (まだ無ければ、その旨を表示します)。

In [ ]:
import json, os
p = os.path.join("lab04_tool_design", "out", "summary_compare.json")
if not os.path.exists(p):
    print("講師の記録はまだありません (本物で回したらここに出ます)")
else:
    s = json.load(open(p, encoding="utf-8"))
    print("model:", s["model"], "/ 問い:", s["n_questions"])
    for v, x in s["variants"].items():
        print(f"  {v:<6} 最初の道具 {x['first_ok']}/{x['n']}  道具なし {x['no_tool']}  入力トークン計 {x['usage_in_total']:,}")